In [ ]:
# ── Step 1: Clone the repository ──────────────────────────────────────────────
!git clone https://github.com/Teja-Jan/Cloud-Cost-Observability-Intelligent-Archival.git
%cd Cloud-Cost-Observability-Intelligent-Archival


In [ ]:
# ── Step 2: Install dependencies (suppress warnings) ──────────────────────────
# Pin cryptography to avoid Colab pre-installed package conflicts
!pip install -q "cryptography>=41,<44"
!pip install -q -r requirements.txt
print("✅ All dependencies installed successfully.")


In [ ]:
# ── Step 3: Launch app via Cloudflared (no password prompt, no JS errors) ─────
import subprocess, time, re

# Download cloudflared binary
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Start Streamlit in the background
!streamlit run src/app.py --server.headless true &>/content/logs.txt &
time.sleep(5)  # Wait for Streamlit to initialize

# Open a Cloudflare tunnel and capture the public URL
proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("⏳ Starting tunnel, please wait...")
for line in proc.stdout:
    line = line.decode('utf-8', errors='replace')
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print(f'\n✅ Cloud Intelligence Platform is live at:\n\n   👉  {url}\n')
        break
